# **US Airline Flight Routes and Fares Market analysis(1993-2024)**
--------------------------------
### **Date**: Feb 7, 2026
### **Author**: Zakarias Musa
--------------------------------
#### **Description**: Explore US domestic airline data to uncover patterns in pricing, demand, competition, market structure, and Time-Series Trend.
#### **Methodology**: Robust handling of skewed data, proper temporal analysis, and justifiable insights without misrepresenting outliers.
--------------------------------

### **I. Data cleaning and preprocessing**

In [ ]:
# Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
airline_data=pd.read_csv("/Users/napi/Documents/us_airline_data/US Airline Flight Routes and Fares 1993-2024.csv")
airline_data.head(5) # return the first five rows

- Our raw data has been successfully loaded to our environment to be analysed

In [ ]:
airline_data.shape # Check the shape of the data

In [ ]:
airline_data.info() #check to understand the structure, data types, and any potential cleaning issues

In [ ]:
airline_data.describe(include='all')
#spotting outliers, empty strings, and default placeholder values like 0 or unknown.

In [ ]:
airline_data.isnull().sum().sort_values(ascending=False) 
# quick snapshot of missing data, sorted so that the columns with the most nulls are on top

- The raw data has a missing values in some of its numerical and categorical columns, so that need to be cleaned for the sake of a cleaned analsis workflow and also to avoid misleaded interpretation.

 -----------------------------------------
### **II. Data Cleaning & Feature Engineering**
 -----------------------------------------

In [ ]:
airline_data0=airline_data.copy() # create a copy of the raw data to avoid data lose

In [ ]:
airline_data0[["fare_low","fare_lg","lf_ms"]]=airline_data0[["fare_low","fare_lg","lf_ms"]].fillna('median')
# fill the three columns with a median value

- The columns fare_low, fare_lg, and lf_ms were imputed using the median because these variables are continuous and likely skewed. The median is robust to outliers and extreme values (common in fare and load factor data), ensuring that the imputation does not distort the overall distribution.

In [ ]:
airline_data0[["carrier_lg","carrier_low","large_ms"]]=airline_data0[["carrier_lg","carrier_low","large_ms"]].fillna('not_reported')
# fill the three categorical column with 'not reported' to retain records while clearly marking unknown entries.

- The categorical columns carrier_lg, carrier_low, and large_ms were filled with "not_reported" to indicate missing information. This preserves the dataset’s structure without introducing misleading values, allowing analysis to handle unknown or missing categories explicitly.

In [ ]:
airline_data0.drop(columns=["Geocoded_City1"], inplace=True)
airline_data0.drop(columns=["Geocoded_City2"], inplace=True)
#drop the the columns with null values and has no any contribution to the analysis
airline_data0.head(5)#quick check if the columns are successfully removed from the copy data

In [ ]:
airline_data0.isnull().sum().sort_values(ascending=False) # Recheck if there exist any null values left after clean up

- The null values are successfully handled all over the existing raws

In [ ]:
airline_data0['tbl'].unique #check for strange values

In [ ]:
airline_data0.drop(columns=["tbl"],inplace=True) # the "tbl" column dropped since it has a constant value in the entire raw
airline_data0.head(5) # shows the first five raws to confirm the "tbl" column was successfully dropped

In [ ]:
# Remove anything inside parentheses like "(Metropolitan Area)"
# Keep them creates duplicate labels for the same market
# "Tampa, Fl (Metropolitan Area)" is not same with "Tampa, Fl"
# For analysis, these must be the SAME market

airline_data0['city1'] = airline_data0['city1'].str.replace(r'\s*\(.*?\)', '', regex=True)
airline_data0['city2'] = airline_data0['city2'].str.replace(r'\s*\(.*?\)', '', regex=True)



In [ ]:
# Split the market name from the state using the LAST comma 
airline_data0[['origin_market', 'origin_state']] = airline_data0['city1'].str.rsplit(',', n=1, expand=True)
airline_data0[['dest_market', 'dest_state']] = airline_data0['city2'].str.rsplit(',', n=1, expand=True)
airline_data0

- Split the market name from the state using the LAST comma as some markets contain multiple cities separated by commas
e.g. "Allentown,Bethlehem,Easton, Pa"

In [ ]:
# Clean the market names
# Remove extra spaces from messy input data
# Standardize capitalization for grouping
# Prevent issues like: "los angeles" , "Los Angeles" , "LOS ANGELES"

for col in ['origin_market', 'dest_market']:
    airline_data0[col] = airline_data0[col].str.strip().str.title()


In [ ]:
# Clean and standardize state codes
for col in ['origin_state', 'dest_state']:
    airline_data0[col] = airline_data0[col].str.strip().str.upper()
airline_data0

In [ ]:
airline_data0.duplicated() #check if there is a duplicated values

In [ ]:
airline_data0 = airline_data0.rename(columns={
    'Year': 'year',
    'citymarketid_1': 'origin_market_id',
    'citymarketid_2': 'destination_market_id',
    'airport_1': 'origin_airport',
    'airport_2': 'destination_airport',
    'airportid_1': 'origin_airport_id',
    'airportid_2': 'destination_airport_id',
    'nsmiles': 'distance_miles',
    'passengers': 'passenger_count',
    'fare': 'average_fare',
    'carrier_lg': 'largest_carrier',
    'large_ms': 'largest_carrier_market_share',
    'fare_lg': 'largest_carrier_fare',
    'carrier_low': 'lowest_fare_carrier',
    'fare_low': 'lowest_carrier_fare',
    'lf_ms': 'market_load_factor',
    'tbl1apk': 'record_id',
    'origin_market': 'origin_market_name',
    'dest_market': 'destination_market_name'
})

# Once created the cleaned ones drop raw city columns as they are redundant and misleading
airline_data0 = airline_data0.drop(columns=['city1', 'city2'])



In [ ]:
airline_data0.columns # A glance, the existing columns

In [ ]:
#changed to the period as we don't know the exact day other than the quarter to convert it to datetime
airline_data0['year_quarter'] = pd.PeriodIndex(
    year=airline_data0['year'],
    quarter=airline_data0['quarter'],
    freq='Q'
)
airline_data0['year_quarter'].dtype #check the data type after conversion to period

In [ ]:
airline_data0[['distance_miles', 'passenger_count',
       'average_fare','largest_carrier', 'largest_carrier_market_share',
       'largest_carrier_fare', 'lowest_fare_carrier', 'market_load_factor',
       'lowest_carrier_fare']].describe(include="all") 
# Inspect the existance of possible outliers on specific numerical columns,  
# top values, and frequency for categorical columns

## **Outlier Analysis**
 -----------------

### **Fare Distribution & Outlier Detection Analysis**
 ------------------

Airline fare distributions are typically **right-skewed**, with a small number of
routes exhibiting extremely high prices due to factors such as low competition,
long distances, or limited demand.

Rather than removing these observations, outliers are **flagged and analyzed
separately** to preserve meaningful market behavior and avoid biasing the analysis.


In [ ]:
# Boxplot is used because:
# It clearly shows median, IQR, and extreme values
# It is robust to skewed distributions
# It visually aligns with the IQR method
airline_data0['average_fare'].plot.box()

### IQR Method for Outlier Detection

Outliers were flagged using the IQR method and analyzed separately, as they represent a small but economically meaningful subset of airline markets rather than erroneous observations.



In [ ]:
# Calculate quartiles for average fare
Q1 = airline_data0['average_fare'].quantile(0.25)
Q3 = airline_data0['average_fare'].quantile(0.75)

# IQR captures the spread of the middle 50% of fares
IQR = Q3 - Q1

# Define both bounds explicitly for methodological correctness
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Flag outliers instead of removing them
airline_data0['is_fare_outlier'] = (
    (airline_data0['average_fare'] < lower_bound) |
    (airline_data0['average_fare'] > upper_bound)
)



In [ ]:
airline_data0['is_fare_outlier'].value_counts()

### Interpretation

  Only 4,757 are flagged as fare outliers out of 245,955, indicating that extreme fares are rare but meaningful. This confirms that the IQR method is not over-flagging and that most airline fares fall within a stable pricing range. The small proportion of outliers suggests these values likely represent structurally different markets (e.g., long-distance routes, low-competition markets, or premium pricing routes) rather than data errors. Therefore, retaining and analyzing these observations separately is justified to avoid biasing average fare and demand analyses.


### Insight from Average fare outlier analysis
The fact that only ~2% of routes exhibit extreme fares suggests that airline pricing is largely competitive for most markets, while a very small subset drives disproportionate price extremes, likely due to monopoly power, limited demand, or operational constraints. These routes are therefore key leverage points for understanding fare inflation rather than noise to be removed.

 -------------------
### **Demand Concentration & High-Traffic Route Analysis**
 -------------------

In [ ]:
# Histogram shows full distribution shape
# Log scale is used because passenger counts are heavily right-skewed
# This prevents high-demand routes from compressing the majority
plt.hist(airline_data0['passenger_count'], bins=50)
plt.xscale('log')  # Log scale reveals structure in skewed distributions
plt.xlabel("Passenger Count (log scale)")
plt.ylabel("Frequency")
plt.title("Distribution of Passenger Counts")
plt.show()



In [ ]:
# IQR is chosen because passenger counts are not normally distributed
# and are often dominated by a small number of very high-demand routes

Q1 = airline_data0['passenger_count'].quantile(0.25)
Q3 = airline_data0['passenger_count'].quantile(0.75)

# IQR represents the typical demand range
IQR = Q3 - Q1

# Define standard statistical bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Flag passenger outliers instead of removing them
# These likely represent peak-demand routes
airline_data0['is_passenger_outlier'] = (
    (airline_data0['passenger_count'] < lower_bound) |
    (airline_data0['passenger_count'] > upper_bound)
)


In [ ]:
# Check proportion of passenger outliers
airline_data0['is_passenger_outlier'].value_counts()


#### Interpretation
Approximately 10% of routes exhibit extreme passenger volumes, indicating a distinct high-demand market segment. Unlike fare outliers (~2%), passenger outliers are relatively common, suggesting that demand concentration is a structural feature of the airline network rather than rare anomalies.

### The insight from passengers outlier analysis
 Passenger traffic is highly concentrated, with a small but substantial subset of routes driving a disproportionate share of total demand.

## **Exploratory Data Analysis**

 -------------------------------
### **I. Market Concentration & Pricing Power Analysis**
 -------------------------------

In [ ]:
# Convert market share to numeric
# coerce errors turns invalid strings into NaN
airline_data0['largest_carrier_market_share'] = pd.to_numeric(
    airline_data0['largest_carrier_market_share'],
    errors='coerce'
)



In [ ]:
airline_data0['largest_carrier_market_share'].describe()


In [ ]:
# Bin market share into competition levels
# Create competition levels based on the dominant carrier's market share
airline_data0['competition_level'] = pd.cut(
    airline_data0['largest_carrier_market_share'],
    bins=[0, 0.4, 0.6, 0.8, 1.0],#Using bins avoids assuming linear effects
    labels=['Highly competitive', 'Moderate', 'Low competition', 'Near monopoly'],
    include_lowest=True #Labels turn numbers into market structure language
)
# Normalize fare by distance to remove route-length bias
airline_data0['fare_per_mile'] = (
    airline_data0['average_fare'] / airline_data0['distance_miles']
)

airline_data0['competition_level']

In [ ]:
# Calculate fare per mile for each competition level
# Aggregates fares within each competition regime
# Mean is appropriate because i'm comparing group-level pricing behavior, not individual routes
airline_data0.groupby('competition_level')['fare_per_mile'].mean()



#### Interpretation
- Fare per mile increases consistently as market concentration rises.
- Highly competitive markets average $0.16 per mile, while near-monopoly markets reach $0.33 per mile.
- That is roughly a 2× increase in price per mile from competitive to monopoly conditions.

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(
    x='competition_level',
    y='fare_per_mile',
    data=airline_data0,
    ci=None
)

plt.title("Fare per Mile Increases as Market Competition Decreases")
plt.xlabel("Competition Level")
plt.ylabel("Fare per Mile ($)")
plt.show()
# Barplot is used because:
# The independent variable (competition_level) is categorical
# The goal is to compare average fare per mile across groups
# Barplots clearly display group-level central tendency (mean)
# They make monotonic trends easy to see


### Market Competition and Fare Levels insight

As market concentration increases, fare per mile rises steadily, nearly doubling from highly competitive to near-monopoly markets.

This indicates that reduced competition significantly increases pricing power, even after controlling for distance.

 -------------------------------
### **II. Economies of Distance in Airline Pricing**
 -------------------------------

In [ ]:
# Create operationally meaningful distance bands
# Reason: fare-distance relationship is non-linear
airline_data0['distance_band'] = pd.cut(
    airline_data0['distance_miles'],
    bins=[0, 500, 1000, 1500, 2500, 4000],
    labels=['Very short', 'Short', 'Medium', 'Long', 'Very long']
)

# Violin plot reveals distribution width, not just central tendency
sns.violinplot(
    x='distance_band',
    y='average_fare',
    data=airline_data0
)

plt.title("Fare Distribution by Distance Band")
plt.xlabel("Distance Category")
plt.ylabel("Average Fare")
plt.show()



While average fares increase with distance, the variance of fares increases much faster than the mean.
This indicates that distance explains baseline costs but fails to explain pricing behavior, especially on long-haul routes.

### Insight
Distance influences minimum airfare levels, but pricing flexibility increases sharply with distance.
Long-haul markets exhibit extreme fare dispersion, indicating that factors beyond cost—such as market power, timing, and route exclusivity—dominate pricing decisions.

 -------------------------------
### **III. Demand Distribution & Price Sensitivity Analysis**
 -------------------------------

In [ ]:
# Inspect passenger volume distribution
# Reason: passenger counts are heavily right-skewed
airline_data0['passenger_count'].plot.hist(bins=40)
plt.title("Distribution of Passenger Volume")
plt.xlabel("Passenger Count")
plt.show()
# histogram was used to see the univariable structure(passenger_count)
# To understand the shape of the distribution
# To check skewness and detect extreme concentration

The histogram shows:
- Passenger volume is heavily right-skewed
- Most routes have low passenger counts
- A small number of routes dominate traffic

Airline markets are highly concentrated in volume.
Average passenger count is misleading due to extreme skewness.

In [ ]:
# Scatter used to show the relationship between the two continuous variables,
# To detect patterns, clustering, or structural constraints
# To test whether demand visibly disciplines pricing
# check whether aggregation is even justified
airline_data0.plot.scatter(
    x='passenger_count',
    y='average_fare'
)
plt.title("Passenger Volume vs Average Fare")
plt.xlabel("Passenger Count")
plt.ylabel("Average Fare")
plt.show()



High-volume markets display price compression rather than price minimization. While extreme fares become less frequent as passenger volume increases, fares do not collapse to low levels. This demonstrates that demand constrains extreme pricing but does not discipline fare levels, contradicting the assumption that economies of scale alone drive lower prices.

#### Insight: passenger volume vs fare
- Demand influences fare stability but does not determine fare level.
Airline pricing remains elevated even in high-traffic markets.

 -------------------------------
### **IV. Capacity Utilization Impact on Pricing Strategy**
 -------------------------------

In [ ]:
airline_data0['market_load_factor'] = pd.to_numeric(
    airline_data0['market_load_factor'],
    errors='coerce'
)
#convert the 'market load factor' values to appropriate numeric value(float)

In [ ]:
airline_data0['market_load_factor'].describe() # To explore the statistical summary on market load factor 

In [ ]:
# Create efficiency bands based on industry benchmarks
airline_data0['load_band'] = pd.cut(
    airline_data0['market_load_factor'],
    bins=[0, 0.6, 0.7, 0.8, 0.9, 1.0],
    labels=['Very low', 'Low', 'Medium', 'High', 'Very high']
)
airline_data0['load_band'] # shows the binned route-quarter

In [ ]:
# Boxplot highlights median and spread clearly when the distribution shape is not the focus
# boxplot is better to answer:
# Does median fare change with load factor?
# Does variability shrink when planes are full?
sns.boxplot(
    x='load_band',
    y='average_fare',
    data=airline_data0
)

plt.title("Fare Distribution by Load Factor")
plt.xlabel("Market Load Factor Category")
plt.ylabel("Average Fare")
plt.show()

- Boxplot shows similar median fares across load factor bands.
- High fares occur in every load category
- No strong upward pattern as load increases

From this we can conclude that Load factor does not strongly determine pricing behavior.

In [ ]:
# A correlation analysis(Heatmap) is used only to confirm weak relationships between the two variables
# It quickly shows strength and direction of linear relationships
# Validates whether relationships seen visually are statistically meaningful
# its used to confirm if load factor is actually correlated with fare
numeric_cols = [
    'distance_miles',
    'passenger_count',
    'market_load_factor',
    'average_fare'
]# include the columns which the relationships has to be confirmed

sns.heatmap(
    airline_data0[numeric_cols].corr(),
    annot=True,
    cmap='coolwarm'
)

plt.title("Correlation Between Key Numerical Variables")
plt.show()


 #### Interpretation
- Load factor vs fare: −0.19 (weak, slightly negative)
- Passenger count vs fare: −0.17 (weak)
- Distance vs fare: 0.50 - moderate (baseline cost effect only)


Operational efficiency and demand have weak linear influence on fares.
Distance has moderate effect, but does not fully explain pricing.
Correlation analysis confirms a weak negative relationship between load factor and fare.


#### Combined Insight:

Airline pricing appears weakly constrained by efficiency or demand metrics, suggesting fares are primarily shaped by broader market structure rather than operational pressure.
- Market Load Factor Has Weak Influence on Airfare Levels

### **V. Time-Series Trend Analysis (1993–2024)** 

In [ ]:
#Average Fare Over Time
#Insight Long-term upward trend 
#Reflects inflation, fuel costs, and market consolidation
fare_trend = airline_data0.groupby('year')['average_fare'].mean()

plt.figure(figsize=(12,6))

plt.plot(fare_trend.index, fare_trend.values)

plt.title("Average Airfare Trend Over Time")

plt.xlabel("Year")

plt.ylabel("Average Fare (USD)")

plt.show()

#### Trend Analysis Interpretation:
- After moderate fluctuations during the 1990s, average airfares declined significantly between 2000 and 2004, reaching their lowest point around 2004–2005.
- Beginning in 2005, fares rebounded strongly and entered a structural growth phase, with a sustained upward trend observed from 2010 onward.
- Between 2012 and 2019, fares stabilized at a higher pricing regime (above $240), indicating possible increased pricing power or rising operating costs.
- A sharp decline occurred in 2020 due to the COVID-19 pandemic, followed by a rapid recovery in subsequent years.
- By 2024, average fares reached their highest observed level, surpassing pre-pandemic peaks.

#### Insight
- The airline market shows long-term fare growth driven by increased pricing power and market concentration, with strong sensitivity to macroeconomic shocks such as COVID-19. Routes with lower competition tend to generate higher revenue efficiency per mile.

#### Business Implication
- Airlines can maximize profitability by strategically focusing on high-demand, low-competition routes while maintaining operational flexibility to withstand external shocks. Route-level competition analysis should guide pricing, expansion, and revenue management decisions.